# CNP-TC V2 - Zero-shot dominant retraining
v1 trained mostly in few-shot mode (p_zero_shot=0.30); zero-shot MAE was 6.24.
v2 flips the mix to p_zero_shot=0.80 so the model is primarily trained
on descriptor-only prediction, with occasional few-shot context.
Expected: zero-shot MAE drop from 6.24 to ~1.2-1.5 (matching STAR).

In [ ]:
EPOCHS = 1200
BATCH_SIZE = 16
LR = 2e-4
N_SEEDS = 5
CTX_NOISE = 0.03
MAX_CTX_POINTS = 32
P_ZERO_SHOT = 0.80
N_CHEB = 64
N_COS = 32
D_MODEL = 256
N_ENC_LAYERS = 6
N_DEC_LAYERS = 3
N_HEADS = 8
N_E_FREQ = 32
N_K_FREQ = 16
LAM_SPEC = 5e-5
LAM_SMOOTH = 5e-4

In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--index-url', 'https://download.pytorch.org/whl/cu121',
                       'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
AUX_DIR = list(INPUT.rglob('combined_data.csv'))[0].parent
CODE_DIR = list(INPUT.rglob('scripts'))[0].parent
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'TaylorCouetteML'
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(CODE_DIR, REPO_DIR)
os.chdir(REPO_DIR)
INPUT_CSV = REPO_DIR / 'data' / 'Input' / 'combined_data.csv'
INPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUX_DIR / 'combined_data.csv', INPUT_CSV)
print('Setup ready, REPO_DIR =', REPO_DIR)

In [ ]:
OUT_DIR = WORK / 'runs' / 'cnp_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)
args = [sys.executable, 'scripts/train_cnp_pro.py',
        '--epochs', str(EPOCHS),
        '--batch', str(BATCH_SIZE),
        '--lr', str(LR),
        '--n_seeds', str(N_SEEDS),
        '--ctx_noise', str(CTX_NOISE),
        '--max_ctx_points', str(MAX_CTX_POINTS),
        '--p_zero_shot', str(P_ZERO_SHOT),
        '--n_cheb', str(N_CHEB),
        '--n_cos', str(N_COS),
        '--d_model', str(D_MODEL),
        '--n_enc_layers', str(N_ENC_LAYERS),
        '--n_dec_layers', str(N_DEC_LAYERS),
        '--n_heads', str(N_HEADS),
        '--n_E_freq', str(N_E_FREQ),
        '--n_k_freq', str(N_K_FREQ),
        '--lam_spec', str(LAM_SPEC),
        '--lam_smooth', str(LAM_SMOOTH),
        '--out_root', str(OUT_DIR)]
print('>>>', ' '.join(args))
t0 = time.time()
rc = subprocess.call(args, cwd=str(REPO_DIR))
print(f'<<< exit={rc} elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'training failed'

In [ ]:
for root, dirs, files in os.walk(OUT_DIR):
    for f in files:
        p = Path(root) / f
        if p.suffix in ['.pt', '.pth', '.json', '.csv', '.npz']:
            print(p.relative_to(WORK), f'({p.stat().st_size/1e6:.2f} MB)')